## 📝 Implement MongoDB context | 🏗️ CBOT PROJECT

#### 📝 Import utilities for data processing and file handling | 🏗️ CBOT PROJECT

In [1]:
import json
import gzip
import uuid
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

#### 📝 Configure input and output paths | 🏗️ CBOT PROJECT

In [2]:
INPUT_PATH = Path("kitsune.kitsune.json")
OUTPUT_PATH = Path("events_flat.jsonl")

#### 📝 Load environment variables and Azure OpenAI configuration | 🏗️ CBOT PROJECT

In [9]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path("/home/xtallet/inari/inari-henzuru-data")
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(dotenv_path=ENV_PATH)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from app.config.settings import AzureOpenAIConfig
cfg = AzureOpenAIConfig()

#### 📝 Create Azure OpenAI client and run chat completion example | 🏗️ CBOT PROJECT

In [10]:
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=cfg.AZURE_OPENAI_API_KEY,
    azure_endpoint=cfg.AZURE_OPENAI_API_ENDPOINT,
    api_version=cfg.AZURE_OPENAI_API_VERSION,
)

# Chat Completion example
resp = client.chat.completions.create(
    model=cfg.AZURE_OPENAI_LLM_DEPLOYMENT_NAME,
    messages=[{"role": "user", "content": "Hello Azure OpenAI, ¿all good?"}],
)
print(resp.choices[0].message.content)

¡Hola! Todo bien por aquí, gracias por preguntar. ¿En qué puedo ayudarte hoy? 😊


#### 📝 Utility to read JSON or JSONL files (with .gz support) | 🏗️ CBOT PROJECT

In [3]:
def iter_json_records(path: Path) -> Iterable[Dict[str, Any]]:
    """
    Yields each JSON object in the file.
    - If the file is a JSON array -> yields each element.
    - Otherwise treats the file as JSON Lines (one JSON object per line).
    - Supports .gz compressed files.
    """
    open_fn = gzip.open if path.suffix == ".gz" else open
    mode = "rt" if path.suffix == ".gz" else "r"

    with open_fn(path, mode, encoding="utf-8") as f:
        # Try to load as a single JSON value first (could be an array)
        try:
            data = json.load(f)
            if isinstance(data, list):
                for obj in data:
                    if isinstance(obj, dict):
                        yield obj
                return
            elif isinstance(data, dict):
                # Single object file (rare for dumps) -> still yield it
                yield data
                return
        except Exception:
            pass

    # If we are here, fallback to JSONL (one object per line)
    with open_fn(path, mode, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if isinstance(obj, dict):
                yield obj

#### 📝 Safe nested access and dictionary flattening utilities | 🏗️ CBOT PROJECT

In [4]:
# Robust normalize_event for your MongoDB dump (English comments)
# - This function does NOT assume a field named "event_name" exists in the raw JSON.
# - It extracts 'manifest' and/or 'metadata.name' when present and keeps them explicit.
# - It returns (id, payload, text) suitable for upserting to Qdrant (payload keys are real/derived from the input).

import uuid
from typing import Any, Dict, Tuple

def safe_get(d: Dict[str, Any], path: str, default=None):
    """
    Safe nested dict access using dot notation.
    Example: safe_get(doc, "data.data.data.policy_id")
    """
    cur = d
    for key in path.split("."):
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key, default)
        if cur is default:
            return default
    return cur

def flatten_dict(d: Dict[str, Any], prefix: str = "", sep: str = ".", max_depth: int = 6):
    """
    Flatten nested dict into a single-level dict with dot-separated keys.
    Lists are JSON-serialized to avoid explosion of fields.
    """
    import json
    out = {}
    if max_depth < 0 or not isinstance(d, dict):
        return out
    for k, v in d.items():
        key = f"{prefix}{sep}{k}" if prefix else str(k)
        if isinstance(v, dict):
            out.update(flatten_dict(v, key, sep=sep, max_depth=max_depth - 1))
        elif isinstance(v, list):
            out[key] = json.dumps(v, ensure_ascii=False)
        else:
            out[key] = v
    return out

#### 📝 Normalize MongoDB event document into structured payload and text | 🏗️ CBOT PROJECT

In [5]:
def normalize_event(doc: Dict[str, Any]) -> Tuple[str, Dict[str, Any], str]:
    """
    Normalize one MongoDB event document into:
      - oid: stable id (from _id.$oid if present, else random uuid)
      - payload: flat dict with real/derived keys (no invented event_name)
      - text: a human-readable summary good for embeddings
    """
    # 1) stable id
    oid = safe_get(doc, "_id.$oid") or str(uuid.uuid4())

    # 2) Try to get "manifest" (example: "PolicyQuoted")
    manifest = safe_get(doc, "data.manifest")

    # 3) Try a few places where metadata name might appear (some documents duplicate metadata)
    metadata_name = (
        safe_get(doc, "data.data.metadata.name")
        or safe_get(doc, "data.data.data.metadata.name")
        or safe_get(doc, "data.metadata.name")
    )

    # 4) Pick a readable 'name' only if present (we keep both manifest and metadata_name explicit)
    name = manifest or metadata_name or None

    # 5) Choose a reasonable 'deep' payload to flatten.
    #    Common structures in your example: data.data.data  (fall back to data.data, then data)
    deep_candidates = ["data.data.data", "data.data", "data"]
    deep = {}
    for path in deep_candidates:
        cand = safe_get(doc, path)
        if isinstance(cand, dict) and cand:
            deep = dict(cand)  # copy to avoid mutating original
            break

    # 6) Extract commonly used fields from the deep payload (with fallbacks)
    occurred_on = safe_get(deep, "occurred_on") or safe_get(doc, "createdAt.$date") or safe_get(doc, "createdAt")
    policy_id = safe_get(deep, "policy_id")
    if not policy_id:
        # fallback: try top-level "keys" list (example: "policy-<uuid>")
        keys = doc.get("keys")
        if isinstance(keys, list) and len(keys) > 0:
            first = keys[0]
            if isinstance(first, str):
                if first.startswith("policy-"):
                    # strip "policy-" prefix to get the UUID-like id
                    policy_id = first.split("policy-", 1)[-1]
                else:
                    policy_id = first

    # 7) Extract metadata block (aggregate_id, author, ip, event_version)
    meta_block = safe_get(doc, "data.data.metadata") or safe_get(doc, "data.data.data.metadata") or {}
    aggregate_id = meta_block.get("aggregate_id")
    event_version = meta_block.get("event_version")
    author_username = meta_block.get("author_username")
    author_fullname = meta_block.get("author_fullname")
    author_id = meta_block.get("author_id")
    ip = meta_block.get("ip")

    # 8) Remove keys we already extracted from deep before flattening
    for k in ("occurred_on", "policy_id"):
        deep.pop(k, None)

    # 9) Flatten the remaining deep payload under "data.*"
    flat_deep = flatten_dict(deep, prefix="data")

    # 10) Build a compact, human-readable 'text' summary (ideal input for embedding)
    lines = []
    if manifest:
        lines.append(f"Manifest: {manifest}")
    if metadata_name and metadata_name != manifest:
        lines.append(f"Metadata name: {metadata_name}")
    if policy_id:
        lines.append(f"Policy ID: {policy_id}")
    if aggregate_id:
        lines.append(f"Aggregate ID: {aggregate_id}")
    if occurred_on:
        lines.append(f"Occurred on: {occurred_on}")
    if author_fullname or author_username:
        a = author_fullname or ""
        u = author_username or ""
        lines.append(f"Author: {a} ({u})".strip())
    if ip:
        lines.append(f"IP: {ip}")
    if event_version is not None:
        lines.append(f"Event version: {event_version}")
    if safe_get(doc, "createdAt.$date") or safe_get(doc, "createdAt"):
        lines.append(f"Inserted in DB: {safe_get(doc, 'createdAt.$date') or safe_get(doc, 'createdAt')}")

    if flat_deep:
        lines.append("Data details:")
        for k, v in flat_deep.items():
            lines.append(f"  - {k}: {v}")

    text = "\n".join(lines)

    # 11) Assemble payload (only real or clearly-derived keys)
    payload = {
        "mongo_id": oid,
        "manifest": manifest,
        "metadata_name": metadata_name,
        "policy_id": policy_id,
        "aggregate_id": aggregate_id,
        "occurred_on": occurred_on,
        "author_username": author_username,
        "author_fullname": author_fullname,
        "author_id": author_id,
        "ip": ip,
        "event_version": event_version,
        "created_at": safe_get(doc, "createdAt.$date") or safe_get(doc, "createdAt"),
        # keep a 'text' field for embeddings too
        "text": text,
    }
    # Merge flattened deep fields (they are namespaced as data.*)
    payload.update(flat_deep)

    return oid, payload, text

#### 📝 Split texts and process embeddings in batches to handle token limits | 🏗️ CBOT PROJECT

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def chunk_texts_for_embeddings(texts: List[str], max_chunk_size: int = 200000) -> List[str]:
    """
    Split texts into chunks that respect token limits for OpenAI embeddings.
    OpenAI's text-embedding-3-small has a limit of ~300k tokens per request.
    We'll be conservative and use 200k as max_chunk_size.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,  # characters per chunk
        chunk_overlap=200,  # overlap between chunks
        length_function=len,  # use character count as approximation
        separators=["\n\n", "\n", ". ", " ", ""]  # split on these in order
    )
    
    all_chunks = []
    current_batch = []
    current_batch_size = 0
    
    for text in texts:
        # Split this text into chunks
        chunks = splitter.split_text(text)
        
        for chunk in chunks:
            # Rough token estimation: ~4 characters per token
            estimated_tokens = len(chunk) // 4
            
            # If adding this chunk would exceed the limit, process current batch
            if current_batch_size + estimated_tokens > max_chunk_size and current_batch:
                all_chunks.extend(current_batch)
                current_batch = [chunk]
                current_batch_size = estimated_tokens
            else:
                current_batch.append(chunk)
                current_batch_size += estimated_tokens
    
    # Add the last batch
    if current_batch:
        all_chunks.extend(current_batch)
    
    return all_chunks

def process_embeddings_in_batches(texts: List[str], batch_size: int = 200000) -> List[List[float]]:
    """
    Process embeddings in batches to avoid token limits.
    Returns a list of embedding vectors.
    """
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    all_embeddings = []
    
    # Split texts into manageable chunks
    chunked_texts = chunk_texts_for_embeddings(texts, batch_size)
    
    print(f"Original {len(texts)} texts split into {len(chunked_texts)} chunks")
    
    # Process in batches
    for i in range(0, len(chunked_texts), 100):  # Process 100 chunks at a time
        batch = chunked_texts[i:i+100]
        print(f"Processing batch {i//100 + 1}/{(len(chunked_texts) + 99)//100} ({len(batch)} chunks)")
        
        try:
            batch_embeddings = embeddings.embed_documents(batch)
            all_embeddings.extend(batch_embeddings)
            print(f"Successfully processed batch {i//100 + 1}")
        except Exception as e:
            print(f"Error processing batch {i//100 + 1}: {e}")
            # If batch fails, try processing individually
            for j, text in enumerate(batch):
                try:
                    single_embedding = embeddings.embed_documents([text])
                    all_embeddings.extend(single_embedding)
                except Exception as e2:
                    print(f"Error processing individual text {i+j}: {e2}")
                    # Add a zero vector as fallback
                    all_embeddings.append([0.0] * 1536)
    
    return all_embeddings

#### 📝 Load MongoDB dump from JSON file into Python | 🏗️ CBOT PROJECT

In [7]:
import json

# Path to your MongoDB dump
json_path = "kitsune.kitsune.json"

# Load the entire file into Python
with open(json_path, "r") as f:
    raw_docs = json.load(f)

print(f"Loaded {len(raw_docs)} documents")
print(type(raw_docs[0]))  # should be dict


Loaded 26885 documents
<class 'dict'>


#### 📝 Normalize MongoDB documents | Extract stable IDs, payloads, and human-readable text

In [8]:
# Normalize every document
normalized = [normalize_event(doc) for doc in raw_docs]

# Each entry is a tuple (oid, payload, text)
print(f"First normalized sample:\n\nID: {normalized[0][0]}\n")
print(f"Payload keys: {list(normalized[0][1].keys())[:10]}\n")
print(f"Text preview:\n{normalized[0][2][:500]}")


First normalized sample:

ID: 669e541d2d8ffcd1725890c4

Payload keys: ['mongo_id', 'manifest', 'metadata_name', 'policy_id', 'aggregate_id', 'occurred_on', 'author_username', 'author_fullname', 'author_id', 'ip']

Text preview:
Manifest: PolicyQuoted
Policy ID: 0e832331-4f80-4982-b3b2-91a3778724cd
Aggregate ID: policy-0e832331-4f80-4982-b3b2-91a3778724cd
Occurred on: 2024-05-22T16:57:06.002000+00:00
Author: Inari Support (inari.support)
IP: 10.101.1.125
Event version: 1
Inserted in DB: 2024-05-22T16:57:06.002Z
Data details:
  - data.metadata.name: PolicyQuoted
  - data.metadata.aggregate_id: policy-0e832331-4f80-4982-b3b2-91a3778724cd
  - data.metadata.event_version: 1
  - data.metadata.author_username: inari.support
 


#### ⚡ Embed MongoDB documents with Azure OpenAI and upsert into Qdrant | 🏗️ CBOT PROJECT

In [15]:
# Use Azure OpenAI embeddings with Qdrant (with batching and rate-limit handling)

import sys
import time
from typing import List
from pathlib import Path
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.http.models import VectorParams, Distance, PointStruct
from langchain_openai import AzureOpenAIEmbeddings

# Load project settings and environment
PROJECT_ROOT = Path("/home/xtallet/inari/inari-henzuru-data")
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(dotenv_path=ENV_PATH)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from app.config.settings import AzureOpenAIConfig
cfg = AzureOpenAIConfig()

# 1) Initialize in-memory Qdrant
client = QdrantClient(":memory:")

# 2) Initialize Azure OpenAI embeddings
emb = AzureOpenAIEmbeddings(
    azure_deployment=cfg.AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME,
    openai_api_version=cfg.AZURE_OPENAI_API_VERSION,
    azure_endpoint=cfg.AZURE_OPENAI_API_ENDPOINT,
    api_key=cfg.AZURE_OPENAI_API_KEY,
)

# Determine embedding dimensionality from the deployment
embedding_dim = len(emb.embed_query("dimension probe"))

# 3) Create the collection with the correct dimensionality
collection_name = "insurance_events"
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
)

# 4) Limit to the first N documents for a quick test
LIMIT_DOCS = 500
normalized_subset = normalized[:LIMIT_DOCS]
print(f"Processing the first {LIMIT_DOCS} documents out of {len(normalized)} total")

# 5) Extract texts from the subset
texts = [text for _, _, text in normalized_subset]

# 6) Generate embeddings with Azure OpenAI (batching + retries to handle rate limits)
def chunk_list(items: List[str], size: int):
    for i in range(0, len(items), size):
        yield items[i:i + size]

BATCH_SIZE = 64         # tune to respect your Calls-Per-Minute
SLEEP_BETWEEN = 1.2     # seconds between requests
MAX_RETRIES = 6

vectors: List[List[float]] = []
for batch in chunk_list(texts, BATCH_SIZE):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            vecs = emb.embed_documents(batch, chunk_size=1024)
            vectors.extend(vecs)
            break
        except Exception as e:
            if attempt == MAX_RETRIES:
                raise
            wait_s = min(60, 2 ** attempt) + (0.5 * attempt)
            print(f"Rate limit or transient error, retrying in {wait_s:.1f}s (attempt {attempt})")
            time.sleep(wait_s)
    time.sleep(SLEEP_BETWEEN)

print(f"Subset documents: {len(normalized_subset)}")
print(f"Generated vectors: {len(vectors)}")

# 7) Upsert into Qdrant (1:1 mapping between texts and vectors)
client.upsert(
    collection_name=collection_name,
    points=[
        PointStruct(
            id=hash(oid) % (2**63),  # Convert ObjectId to a unique positive integer
            vector=vec,
            payload=payload,
        )
        for (oid, payload, _), vec in zip(normalized_subset, vectors)
    ],
)

print(f"✅ {len(vectors)} documents have been embedded with Azure OpenAI and stored in Qdrant.")

/tmp/ipykernel_55267/3819592692.py:38: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Processing the first 500 documents out of 26885 total
Subset documents: 500
Generated vectors: 500
✅ 500 documents have been embedded with Azure OpenAI and stored in Qdrant.


#### 🔍 Query Qdrant with Azure OpenAI embeddings | 🏗️ CBOT PROJECT

In [22]:
# Query Qdrant using Azure OpenAI embeddings

from langchain_openai import AzureOpenAIEmbeddings

# Ensure Azure config is available (reuse if already defined)
try:
    cfg  # type: ignore[name-defined]
except NameError:
    import sys
    from pathlib import Path
    from dotenv import load_dotenv
    PROJECT_ROOT = Path("/home/xtallet/inari/inari-henzuru-data")
    ENV_PATH = PROJECT_ROOT / ".env"
    load_dotenv(dotenv_path=ENV_PATH)
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.append(str(PROJECT_ROOT))
    from app.config.settings import AzureOpenAIConfig
    cfg = AzureOpenAIConfig()

# Reuse embeddings instance if exists; otherwise create it
try:
    emb  # type: ignore[name-defined]
except NameError:
    emb = AzureOpenAIEmbeddings(
        azure_deployment=cfg.AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME,
        openai_api_version=cfg.AZURE_OPENAI_API_VERSION,
        azure_endpoint=cfg.AZURE_OPENAI_API_ENDPOINT,
        api_key=cfg.AZURE_OPENAI_API_KEY,
    )

# Build query vector with the same Azure deployment used for indexing
query = "How many policies do I have in status draf ?"
qvec = emb.embed_query(query)

# Search in Qdrant
results = client.search(
    collection_name=collection_name,
    query_vector=qvec,
    limit=3,
)

for r in results:
    print(r.payload.get("text", r.payload))

Manifest: PolicySubStatusPremiumSet
Policy ID: d9acd393-0f00-4f2d-a65c-7f81c08d39bb
Aggregate ID: policy-d9acd393-0f00-4f2d-a65c-7f81c08d39bb
Occurred on: 2024-05-22T16:35:43.253000+00:00
Author: Inari Support (inari.support)
IP: 10.101.1.93
Event version: 1
Inserted in DB: 2024-05-22T16:35:43.253Z
Data details:
  - data.metadata.name: PolicySubStatusPremiumSet
  - data.metadata.aggregate_id: policy-d9acd393-0f00-4f2d-a65c-7f81c08d39bb
  - data.metadata.event_version: 1
  - data.metadata.author_username: inari.support
  - data.metadata.author_fullname: Inari Support
  - data.metadata.author_id: 9001
  - data.metadata.ip: 10.101.1.93
  - data.metadata.date: 2024-05-22T16:35:43.253195+00:00
  - data.correlation_id: None
Manifest: PolicySubStatusPremiumSet
Policy ID: d7b6c5ab-651a-49b1-8096-e1daf430bfb1
Aggregate ID: policy-d7b6c5ab-651a-49b1-8096-e1daf430bfb1
Occurred on: 2024-05-22T18:22:54.097000+00:00
Author: Inari Support (inari.support)
IP: 10.101.2.244
Event version: 1
Inserted in 

/tmp/ipykernel_55267/1822536721.py:36: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = client.search(


#### 💬 Use Azure OpenAI Chat Model with LangChain | 🏗️ CBOT PROJECT

In [23]:
from langchain_openai import AzureChatOpenAI

# Ensure Azure config is available (reuse if already defined)
try:
    cfg  # type: ignore[name-defined]
except NameError:
    import sys
    from pathlib import Path
    from dotenv import load_dotenv
    PROJECT_ROOT = Path("/home/xtallet/inari/inari-henzuru-data")
    ENV_PATH = PROJECT_ROOT / ".env"
    load_dotenv(dotenv_path=ENV_PATH)
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.append(str(PROJECT_ROOT))
    from app.config.settings import AzureOpenAIConfig
    cfg = AzureOpenAIConfig()

# Initialize Azure Chat LLM
llm = AzureChatOpenAI(
    azure_deployment=cfg.AZURE_OPENAI_LLM_DEPLOYMENT_NAME,
    openai_api_version=cfg.AZURE_OPENAI_API_VERSION,
    azure_endpoint=cfg.AZURE_OPENAI_API_ENDPOINT,
    api_key=cfg.AZURE_OPENAI_API_KEY,
    temperature=0,
)

# Build prompt with retrieved context
context = "\n\n".join([r.payload.get("text", "") for r in results])
prompt = f"""
You are a helpful assistant who answers strictly based on the provided context. Do not use any external knowledge.

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)
print("Answer:", response.content)

BadRequestError: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}